# Batch Prediction for Test Set



## Enter the MMSegmentation main directory

In [1]:
import os
os.chdir('mmsegmentation')

In [2]:
os.getcwd()

'/home/featurize/work/Xylem/20230816/mmsegmentation'

## Import Packages

In [3]:
import os
import numpy as np
import cv2
from tqdm import tqdm

from mmseg.apis import init_model, inference_model, show_result_pyplot
import mmcv

import matplotlib.pyplot as plt
%matplotlib inline

## Load Model

In [6]:
# 模型 config 配置文件
config_file = '/home/featurize/work/Xylem/20230816/mmsegmentation/work_dirs/ZihaoDataset-Mask2Former/ZihaoDataset_Mask2Former_2025.03.07.py'

checkpoint_file = '/home/featurize/work/Xylem/20230816/mmsegmentation/work_dirs/ZihaoDataset-Mask2Former/best_mIoU_iter_11000.pth'

# 计算硬件
# device = 'cpu'
device = 'cuda:0'

In [7]:
model = init_model(config_file, checkpoint_file, device=device)

Loads checkpoint by local backend from path: /home/featurize/work/Xylem/20230816/mmsegmentation/work_dirs/ZihaoDataset-Mask2Former/best_mIoU_iter_11000.pth


## Specify color schemes for each category

In [8]:
# 每个类别的 BGR 配色
palette = [
    ['background', [0,0,0]],
    ['A', [0,0,127]],
    ['B', [0,127,0]],
    ['C', [0,127,127]],
    ['D', [127,0,0]],
    ]

palette_dict = {}
for idx, each in enumerate(palette):
    palette_dict[idx] = each[1]

In [9]:
palette_dict

{0: [0, 0, 0], 1: [255, 255, 255]}

## Specify the test set path (can also be changed to the folder path of images to be tested)

In [15]:
PATH_IMAGE = '/home/featurize/work/Xylem/20230816/Xylem_Area/PZY/cle26ab20倍束中横切原图'

In [16]:
os.chdir(PATH_IMAGE)

## Single Image Prediction Function

In [17]:
opacity = 0 # 透明度，越大越接近原图

In [18]:
def process_single_img(img_path, save=False):
    
    img_bgr = cv2.imread(img_path)

    # 语义分割预测
    result = inference_model(model, img_bgr)
    pred_mask = result.pred_sem_seg.data[0].cpu().numpy()

    # 将预测的整数ID，映射为对应类别的颜色
    pred_mask_bgr = np.zeros((pred_mask.shape[0], pred_mask.shape[1], 3))
    for idx in palette_dict.keys():
        pred_mask_bgr[np.where(pred_mask==idx)] = palette_dict[idx]
    pred_mask_bgr = pred_mask_bgr.astype('uint8')

    # 将语义分割预测图和原图叠加显示
    pred_viz = cv2.addWeighted(img_bgr, opacity, pred_mask_bgr, 1-opacity, 0)
    
    # 保存图像至 
    if save:
        save_path = os.path.join('/home/featurize/work/Xylem/20230816/Xylem_Area/PZY/PZY20X结果/'+img_path.split('/')[-1])
        cv2.imwrite(save_path, pred_viz)


## Batch Prediction for Test Set

In [19]:
for each in tqdm(os.listdir()):
    process_single_img(each, save=True)

100%|██████████| 30/30 [00:40<00:00,  1.35s/it]


预测结果保存在 `mmsegmentation/outputs/testset-pred` 目录下

In [14]:
os.chdir(os.path.join('../','../','../'))

## Delete unnecessary files automatically generated by the system

### View redundant files to be deleted

In [55]:
!find . -iname '__MACOSX'

In [56]:
!find . -iname '.DS_Store'

In [57]:
!find . -iname '.ipynb_checkpoints'

./.ipynb_checkpoints


### Delete unnecessary files

In [58]:
!for i in `find . -iname '__MACOSX'`; do rm -rf $i;done

In [59]:
!for i in `find . -iname '.DS_Store'`; do rm -rf $i;done

In [60]:
!for i in `find . -iname '.ipynb_checkpoints'`; do rm -rf $i;done

### Verify that redundant files have been deleted

In [ ]:
!find . -iname '__MACOSX'

In [ ]:
!find . -iname '.DS_Store'

In [ ]:
!find . -iname '.ipynb_checkpoints'

### Calculate the area under a 40X magnification scope

In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image


def count_target_pixels(image_array, target_color, tolerance=0):
    """
    统计图像中目标颜色的像素数量
    """
    target_red, target_green, target_blue = target_color

    target_pixels = np.sum(
        (image_array[:, :, 0] >= target_red - tolerance) & (image_array[:, :, 0] <= target_red + tolerance) &
        (image_array[:, :, 1] >= target_green - tolerance) & (image_array[:, :, 1] <= target_green + tolerance) &
        (image_array[:, :, 2] >= target_blue - tolerance) & (image_array[:, :, 2] <= target_blue + tolerance)
    )

    return int(target_pixels)


def count_other_pixels(image_array, excluded_colors, tolerance=0):
    """
    统计非指定颜色（即非黑色 [0,0,0] 和非目标颜色 target_color）的像素数量
    """
    excluded_masks = []
    for color in excluded_colors:
        red, green, blue = color
        mask = (
            (image_array[:, :, 0] >= red - tolerance) & (image_array[:, :, 0] <= red + tolerance) &
            (image_array[:, :, 1] >= green - tolerance) & (image_array[:, :, 1] <= green + tolerance) &
            (image_array[:, :, 2] >= blue - tolerance) & (image_array[:, :, 2] <= blue + tolerance)
        )
        excluded_masks.append(mask)

    # 创建一个包含所有排除颜色的总掩码
    excluded_mask = np.logical_or.reduce(excluded_masks)

    # 统计不在排除列表中的像素数
    return int(np.sum(~excluded_mask))


def calculate_percentage(part_pixels, total_pixels):
    """
    计算像素占比（%）
    """
    if total_pixels == 0:
        return 0.0
    return (part_pixels / total_pixels) * 100.0


def process_folder_percentage(
    input_folder,
    output_csv_1,
    output_csv_2,
    target_color,
    tolerance=0,
    pixel_to_cm_ratio=0.0188686
):
    """
    处理文件夹中的图片，分别计算：
    1) 目标颜色 target_color 的面积百分比
    2) 非 [0,0,0] 和非 target_color 的面积百分比
    并分别保存到两个 CSV 文件
    """
    results_target = []
    results_other = []

    # 每个像素对应的面积（单位：平方厘米）
    pixel_area_cm2 = pixel_to_cm_ratio ** 2

    # 排除的颜色列表（黑色 [0,0,0] 和目标颜色 target_color）
    excluded_colors = [(0, 0, 0), tuple(target_color)]

    # 若输出目录不存在，自动创建
    for out_path in (output_csv_1, output_csv_2):
        out_dir = os.path.dirname(out_path)
        if out_dir and (not os.path.exists(out_dir)):
            os.makedirs(out_dir, exist_ok=True)

    for filename in os.listdir(input_folder):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.tiff')):
            image_path = os.path.join(input_folder, filename)

            try:
                # 打开图像并转换为 NumPy 数组
                img = Image.open(image_path).convert('RGBA')
                image_array = np.array(img)

                image_width, image_height = img.size
                total_pixels = int(image_width * image_height)

                # 统计目标颜色像素数
                target_pixels = count_target_pixels(image_array, target_color, tolerance)

                # 统计非黑色且非目标颜色的像素数
                other_pixels = count_other_pixels(image_array, excluded_colors, tolerance)

                # 计算百分比
                percentage_target = calculate_percentage(target_pixels, total_pixels)
                percentage_other = calculate_percentage(other_pixels, total_pixels)

                # 计算真实面积（单位：平方厘米）
                real_area_target_cm2 = target_pixels * pixel_area_cm2
                real_area_other_cm2 = other_pixels * pixel_area_cm2

                # 目标颜色统计
                results_target.append({
                    'Image Name': filename,
                    'Image Width (pixels)': image_width,
                    'Image Height (pixels)': image_height,
                    'Total Pixels': total_pixels,
                    'A Pixels': target_pixels,
                    'Real Area (cm^2)': real_area_target_cm2,
                    'Percentage (%)': percentage_target
                })

                # 其他颜色统计
                results_other.append({
                    'Image Name': filename,
                    'Image Width (pixels)': image_width,
                    'Image Height (pixels)': image_height,
                    'Total Pixels': total_pixels,
                    'BCD Pixels': other_pixels,
                    'Real Area (cm^2)': real_area_other_cm2,
                    'Percentage (%)': percentage_other
                })

            except Exception as e:
                print(f"无法处理图片 {filename}: {e}")

    # 保存到 CSV（utf-8-sig：Excel 打开中文不容易乱码）
    df_target = pd.DataFrame(results_target)
    df_target.to_csv(output_csv_1, index=False, encoding='utf-8-sig')

    df_other = pd.DataFrame(results_other)
    df_other.to_csv(output_csv_2, index=False, encoding='utf-8-sig')

    print(f"目标颜色统计已保存到 {output_csv_1}")
    print(f"其他颜色统计已保存到 {output_csv_2}")


# ===== 示例使用 =====
if __name__ == "__main__":
    input_folder = r'../Output'                 # 输入图片文件夹路径
    output_csv_target = r'../Output/TypeI.csv'  # 目标颜色统计
    output_csv_other = r'../Output/TypeII-IV.csv'  # 其他颜色统计

    target_color = [127, 0, 0]   # 目标颜色
    tolerance = 2                # 容差范围
    pixel_to_cm_ratio = 20 / 234 # 20x 放大倍率（按你原来的参数保留）

    process_folder_percentage(
        input_folder,
        output_csv_target,
        output_csv_other,
        target_color,
        tolerance,
        pixel_to_cm_ratio
    )
